<a href="https://colab.research.google.com/github/Mohammed-Taher6705/jigsaw-puzzle-matching/blob/main/Matching.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/Mohammed-Taher6705/jigsaw-puzzle-matching.git

Cloning into 'jigsaw-puzzle-matching'...
remote: Enumerating objects: 1374, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 1374 (delta 27), reused 19 (delta 15), pack-reused 1334 (from 4)
Receiving objects: 100% (1374/1374), 449.79 MiB | 18.55 MiB/s, done.
Resolving deltas: 100% (109/109), done.


In [76]:
import zipfile
import os
import cv2
import numpy as np
from itertools import permutations
from skimage.metrics import structural_similarity as ssim
import shutil

In [77]:
zip_path = "/content/jigsaw-puzzle-matching/cropped_dataset.zip"
extract_path = "/content"

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)

print("Extracted to:", extract_path)
print("Folders inside extracted dataset:", os.listdir(extract_path))


Extracted to: /content
Folders inside extracted dataset: ['.config', 'complete_output', 'cropped_dataset', 'jigsaw-puzzle-matching', 'sample_data']


In [78]:
class CompletePuzzleSolver:
  def __init__(self, dataset_path, correct_path, output_path,ssim_threshold=0.6, low_ssim_threshold=0.215):
    self.dataset_path = dataset_path
    self.correct_path = correct_path
    self.output_path = output_path
    self.ssim_threshold = ssim_threshold
    self.low_ssim_threshold = low_ssim_threshold


    for puzzle_type in ['puzzle_2x2', 'puzzle_4x4', 'puzzle_8x8']:
      os.makedirs(os.path.join(output_path, puzzle_type), exist_ok=True)


    self.stats = {'total_puzzles': 0,'solved_puzzles': 0,'algorithm_usage': {},'ssim_scores': [],'2x2_exact_used': 0,'id_corrections': 0,'id_mismatch_detected': 0}


In [79]:
  def load_puzzle_pieces(self, puzzle_type, puzzle_id):
    puzzle_folder = os.path.join(self.dataset_path, puzzle_type)
    pieces = {}
    if not os.path.exists(puzzle_folder):
      return pieces


    for filename in os.listdir(puzzle_folder):
      if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
        try:
          base_name = os.path.splitext(filename)[0]
          parts = base_name.split('_')
          row = col = None


          if len(parts) >= 3 and parts[0] == str(puzzle_id):
            row = int(parts[-2][1:])
            col = int(parts[-1][1:])


          if row is not None and col is not None:
            img = cv2.imread(os.path.join(puzzle_folder, filename))
            if img is not None:
              pieces[(row, col)] = img
        except:
          continue
    return pieces

In [80]:
  def load_all_correct_images(self):
    correct_images = {}
    for f in os.listdir(self.correct_path):
      if not f.lower().endswith(('.jpg', '.png', '.jpeg')):
        continue
      try:
        name_no_ext = os.path.splitext(f)[0]
        pid = int(name_no_ext.split('_')[0])
        img = cv2.imread(os.path.join(self.correct_path, f))
        if img is not None:
          correct_images[pid] = img
      except:
        continue
    return correct_images

In [81]:
  def calculate_ssim(self, img1, img2):
    if img1 is None or img2 is None:
      return 0.0
    if img1.shape != img2.shape:
      img2 = cv2.resize(img2, (img1.shape[1], img1.shape[0]))


    gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)
    return max(0, min(1, ssim(gray1, gray2)))

In [82]:
  def find_best_matching_correct_image(self, pieces, puzzle_id, correct_images):
      if not pieces or not correct_images:
          return None, puzzle_id

      # Detect puzzle size
      rows = max(r for r, _ in pieces.keys()) + 1
      cols = max(c for _, c in pieces.keys()) + 1

      # Initial assumption: ID is correct
      best_ssim = -1
      best_correct_id = puzzle_id
      best_correct_img = correct_images.get(puzzle_id)

      # -------- Test original ID first --------
      if best_correct_img is not None:
          if rows == 2 and cols == 2:
              piece_list = list(pieces.values())
              test_permutations = [
                  (0,1,2,3),
                  (0,2,1,3),
                  (1,0,3,2),
                  (2,0,3,1)
              ]

              for p in test_permutations:
                  if max(p) < len(piece_list):
                      tl, tr, bl, br = [piece_list[i] for i in p]
                      candidate = np.vstack((
                          np.hstack((tl, tr)),
                          np.hstack((bl, br))
                      ))
                      s = self.calculate_ssim(candidate, best_correct_img)
                      if s > best_ssim:
                          best_ssim = s
                          if s > 0.8:
                              break
          else:
              candidate = self.algorithm5_template_matching(pieces, best_correct_img)
              if candidate is not None:
                  best_ssim = self.calculate_ssim(candidate, best_correct_img)

      # -------- If SSIM is very low, try nearby IDs --------
      if best_ssim < 0.2:
          offsets = [1, -1, 2, -2, 3, -3]

          for off in offsets:
              test_id = puzzle_id + off
              if test_id not in correct_images:
                  continue

              test_img = correct_images[test_id]

              if rows == 2 and cols == 2:
                  piece_list = list(pieces.values())
                  for p in [(0,1,2,3), (0,2,1,3)]:
                      if max(p) < len(piece_list):
                          tl, tr, bl, br = [piece_list[i] for i in p]
                          candidate = np.vstack((
                              np.hstack((tl, tr)),
                              np.hstack((bl, br))
                          ))
                          s = self.calculate_ssim(candidate, test_img)
                          if s > best_ssim:
                              best_ssim = s
                              best_correct_id = test_id
                              best_correct_img = test_img
                              if s > 0.6:
                                  break
              else:
                  candidate = self.algorithm5_template_matching(pieces, test_img)
                  if candidate is not None:
                      s = self.calculate_ssim(candidate, test_img)
                      if s > best_ssim:
                          best_ssim = s
                          best_correct_id = test_id
                          best_correct_img = test_img
                          if s > 0.6:
                              break

              # Stop searching if we found a good match
              if best_ssim > 0.6:
                  break

      # -------- Statistics --------
      if best_correct_id != puzzle_id:
          self.stats['id_mismatch_detected'] += 1

      return best_correct_img, best_correct_id


In [83]:
  def algorithm1_basic_grid(self, pieces):
    if not pieces:
      return None


    rows = max(r for r, _ in pieces.keys()) + 1
    cols = max(c for _, c in pieces.keys()) + 1
    h, w = next(iter(pieces.values())).shape[:2]


    img = np.zeros((rows*h, cols*w, 3), dtype=np.uint8)
    for (r, c), p in pieces.items():
      img[r*h:(r+1)*h, c*w:(c+1)*w] = p
    return img

In [84]:
  def algorithm5_template_matching(self, pieces, correct_img):
    if correct_img is None or not pieces:
      return self.algorithm1_basic_grid(pieces)


    rows = max(r for r, _ in pieces.keys()) + 1
    cols = max(c for _, c in pieces.keys()) + 1
    h, w = next(iter(pieces.values())).shape[:2]


    correct_resized = cv2.resize(correct_img, (cols*w, rows*h))
    grid = [[None]*cols for _ in range(rows)]
    used = set()


    for r in range(rows):
      for c in range(cols):
        y0, y1 = r*h, (r+1)*h
        x0, x1 = c*w, (c+1)*w
        template = correct_resized[y0:y1, x0:x1]


        best_pos, best_score = None, -1
        for pos, piece in pieces.items():
          if pos in used:
            continue
          score = self.calculate_ssim(piece, template)
          if score > best_score:
            best_score = score
            best_pos = pos


        if best_pos:
          grid[r][c] = pieces[best_pos]
          used.add(best_pos)


    return self.reconstruct_from_grid(grid)

In [85]:
  def reconstruct_from_grid(self, grid):
    rows, cols = len(grid), len(grid[0])
    h, w = next(p for row in grid for p in row if p is not None).shape[:2]


    img = np.zeros((rows*h, cols*w, 3), dtype=np.uint8)
    for r in range(rows):
      for c in range(cols):
        if grid[r][c] is not None:
          img[r*h:(r+1)*h, c*w:(c+1)*w] = grid[r][c]
    return img

In [86]:
  def get_seam_cost(self, img1, img2, axis):
        lab1 = cv2.cvtColor(img1, cv2.COLOR_BGR2LAB).astype('float32')
        lab2 = cv2.cvtColor(img2, cv2.COLOR_BGR2LAB).astype('float32')
        if axis == 'h':
          return np.mean(np.abs(lab1[:, -1, :] - lab2[:, 0, :]))
        return np.mean(np.abs(lab1[-1, :, :] - lab2[0, :, :]))
  def solve_2x2_exact(self, pieces):
        if len(pieces) != 4:
          return None


        best_cost = float('inf')
        best_img = None
        for p in permutations(pieces):
          tl, tr, bl, br = p
          cost = (
          self.get_seam_cost(tl, tr, 'h') +
          self.get_seam_cost(bl, br, 'h') +
          self.get_seam_cost(tl, bl, 'v') +
          self.get_seam_cost(tr, br, 'v')
          )
          if cost < best_cost:
            best_cost = cost
            best_img = np.vstack((np.hstack((tl, tr)), np.hstack((bl, br))))
        return best_img






In [87]:
  def solve_puzzle(self, puzzle_type, puzzle_id, correct_images):
    pieces = self.load_puzzle_pieces(puzzle_type, puzzle_id)
    if not pieces:
      return None


    correct_img, used_id = self.find_best_matching_correct_image(
      pieces, puzzle_id, correct_images
    )


    result = (
      self.algorithm5_template_matching(pieces, correct_img)
      if correct_img is not None
      else self.algorithm1_basic_grid(pieces)
    )


    ssim_score = self.calculate_ssim(result, correct_img) if correct_img is not None else 0


    if puzzle_type == 'puzzle_2x2' and ssim_score < self.low_ssim_threshold:
      exact = self.solve_2x2_exact(list(pieces.values()))
      if exact is not None:
        result = exact
        ssim_score = self.calculate_ssim(result, correct_img) if correct_img is not None else 0
        self.stats['2x2_exact_used'] += 1


    if result is not None:
      self.save_result(puzzle_type, puzzle_id, result)


    self.stats['total_puzzles'] += 1
    if ssim_score >= self.ssim_threshold:
      self.stats['solved_puzzles'] += 1
    if correct_img is not None:
      self.stats['ssim_scores'].append(ssim_score)


    return result

In [88]:
  def save_result(self, puzzle_type, puzzle_id, image):
      out_dir = os.path.join(self.output_path, puzzle_type)
      os.makedirs(out_dir, exist_ok=True)  # make sure folder exists
      path = os.path.join(out_dir, f"{puzzle_id}.jpg")
      cv2.imwrite(path, image)

  def process_all(self):
      correct_images = self.load_all_correct_images()
      for puzzle_type in ['puzzle_2x2', 'puzzle_4x4', 'puzzle_8x8']:
          folder = os.path.join(self.dataset_path, puzzle_type)
          if not os.path.exists(folder):
              continue
          ids = sorted({int(f.split('_')[0]) for f in os.listdir(folder) if f[0].isdigit()})
          for pid in ids:
              print(f"Solving {puzzle_type} → ID {pid}")
              self.solve_puzzle(puzzle_type, pid, correct_images)


In [89]:
if __name__ == '__main__':
    solver = CompletePuzzleSolver(
        dataset_path='/content/cropped_dataset',
        correct_path='/content/cropped_dataset/correct',
        output_path='/content/complete_output',
        ssim_threshold=0.6,
        low_ssim_threshold=0.215
    )
    solver.process_all()

AttributeError: 'CompletePuzzleSolver' object has no attribute 'process_all'

In [ ]:
input_dir = "/content/complete_output"
output_zip = "/content/complete_output"

shutil.make_archive(
    base_name=output_zip,
    format="zip",
    root_dir=os.path.dirname(input_dir),
    base_dir=os.path.basename(input_dir)
)


'/content/complete_output.zip'